In [1]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
# Directories 
# ROOT_DIR = Path("/Users/bmacedo/Desktop/final_WM")
ROOT_DIR = Path("/mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome")
HARMONIZED_DIR = ROOT_DIR / "output_data" / "harmonized"
OUTPUT_DIR = ROOT_DIR / "output_data" / "results" 
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load harmonized data

# ---- MAIN DATA ----
df_harmonized_exposome_A = pd.read_csv(HARMONIZED_DIR / "df_harmonized_exposome_A_12_10.csv")
df_harmonized_exposome_B = pd.read_csv(HARMONIZED_DIR / "df_harmonized_exposome_B_12_10.csv")

# ---- COGNITION ----
df_harmonized_exposome_A_cog = pd.read_csv(HARMONIZED_DIR / "df_harmonized_exposome_cognition_A_12_10.csv")
df_harmonized_exposome_B_cog = pd.read_csv(HARMONIZED_DIR / "df_harmonized_exposome_cognition_B_12_10.csv")

# ---- EDUCATION/INCOME SENSITIVITY ----
df_harmonized_exposome_A_sens = pd.read_csv(HARMONIZED_DIR / "df_income_sens_A_11_30.csv")
df_harmonized_exposome_B_sens = pd.read_csv(HARMONIZED_DIR / "df_income_sens_B_11_30.csv")

# ---- Logging ----
print("[MAIN] A:", df_harmonized_exposome_A.shape)
print("[MAIN] B:", df_harmonized_exposome_B.shape)
print("[COG] A:", df_harmonized_exposome_A_cog.shape)
print("[COG] B:", df_harmonized_exposome_B_cog.shape)
print("[SENS] A:", df_harmonized_exposome_A_sens.shape)
print("[SENS] B:", df_harmonized_exposome_B_sens.shape)

[MAIN] A: (8174, 5352)
[MAIN] B: (8174, 5352)
[COG] A: (7638, 5356)
[COG] B: (7638, 5356)
[SENS] A: (6222, 5348)
[SENS] B: (6222, 5348)


In [3]:
MODEL_PREFIXES = {"DKI": "DKI_", "NODDI": "NODDI_", "MAPMRI": "MAPMRI_", "GQI": "GQI_"}

def split_msmt(col):
    if not col.startswith("bundle_"):
        return None, None

    parts = col[len("bundle_"):].split("_")
    # bundle = first two parts
    bundle = "_".join(parts[:2])

    # metric = everything else
    metric = "_".join(parts[2:])
    return bundle, metric

def model_of_metric(metric):
    # Returns one of {'DKI','NODDI','MAPMRI','GQI','macro'}
    for model, pref in MODEL_PREFIXES.items():
        if metric.startswith(pref): return model
    return "macro"  # msmt_* but no micro prefix -> macrostructure

def column_kind(col):
    # Classify a column: 'covariate' if not msmt_*, otherwise one of {'DKI','NODDI','MAPMRI','GQI','macro'}
    bundle, metric = split_msmt(col)
    if bundle is None: return "covariate"
    return model_of_metric(metric)

def catalog_models(df):
    return pd.Series({c: column_kind(c) for c in df.columns}, name="kind")

def filter_df_by_models(df, include_models=("DKI",), keep_macro=True, keep_covariates=True):
    # keeps all covariates if keep_covariates, all macrostructure metrics if keep_macro, micro metrics only from include_models
    keep_cols = []
    for c in df.columns:
        kind = column_kind(c)
        if kind == "covariate" and keep_covariates: keep_cols.append(c)
        elif kind == "macro" and keep_macro: keep_cols.append(c)
        elif kind in include_models: keep_cols.append(c)
    return df.loc[:, keep_cols]

In [4]:
def save_filtered(df, *, include_models, keep_macro, keep_covariates, fname):
    filtered_df = filter_df_by_models(df, include_models=include_models, keep_macro=keep_macro, keep_covariates=keep_covariates)
    filtered_df.to_csv(OUTPUT_DIR / fname, index=False)
    print(f"[SAVE] {fname}: {filtered_df.shape}")

# label, include_models, keep_macro
specs = [("macro", (), True), ("NODDI", ("NODDI",), False), ("macro_plus_NODDI", ("NODDI",), True), 
         ("DKI", ("DKI",), False), ("macro_plus_DKI", ("DKI",), True)]

for label, models, keep_macro in specs:
    # ---- MAIN ----
    save_filtered(df_harmonized_exposome_A, include_models=models, keep_macro=keep_macro, keep_covariates=True, 
                  fname=f"dfA_{label}_12_10.csv")
    save_filtered(df_harmonized_exposome_B, include_models=models, keep_macro=keep_macro, keep_covariates=True, 
              fname=f"dfB_{label}_12_10.csv")

    # ---- COGNITION ----
    save_filtered(df_harmonized_exposome_A_cog, include_models=models, keep_macro=keep_macro, keep_covariates=True,
                  fname=f"dfA_{label}_cog_12_10.csv")
    save_filtered(df_harmonized_exposome_B_cog, include_models=models, keep_macro=keep_macro, keep_covariates=True,
                  fname=f"dfB_{label}_cog_12_10.csv")

    # ---- SENSITIVITY ----
    save_filtered(df_harmonized_exposome_A_sens, include_models=models, keep_macro=keep_macro, keep_covariates=True, 
                  fname=f"dfA_{label}_sens_11_30.csv")
    save_filtered(df_harmonized_exposome_B_sens, include_models=models, keep_macro=keep_macro, keep_covariates=True, 
                  fname=f"dfB_{label}_sens_11_30.csv")

[SAVE] dfA_macro_12_10.csv: (8174, 1136)
[SAVE] dfB_macro_12_10.csv: (8174, 1136)
[SAVE] dfA_macro_cog_12_10.csv: (7638, 1140)
[SAVE] dfB_macro_cog_12_10.csv: (7638, 1140)
[SAVE] dfA_macro_sens_11_30.csv: (6222, 1132)
[SAVE] dfB_macro_sens_11_30.csv: (6222, 1132)
[SAVE] dfA_NODDI_12_10.csv: (8174, 392)
[SAVE] dfB_NODDI_12_10.csv: (8174, 392)
[SAVE] dfA_NODDI_cog_12_10.csv: (7638, 396)
[SAVE] dfB_NODDI_cog_12_10.csv: (7638, 396)
[SAVE] dfA_NODDI_sens_11_30.csv: (6222, 388)
[SAVE] dfB_NODDI_sens_11_30.csv: (6222, 388)
[SAVE] dfA_macro_plus_NODDI_12_10.csv: (8174, 1508)
[SAVE] dfB_macro_plus_NODDI_12_10.csv: (8174, 1508)
[SAVE] dfA_macro_plus_NODDI_cog_12_10.csv: (7638, 1512)
[SAVE] dfB_macro_plus_NODDI_cog_12_10.csv: (7638, 1512)
[SAVE] dfA_macro_plus_NODDI_sens_11_30.csv: (6222, 1504)
[SAVE] dfB_macro_plus_NODDI_sens_11_30.csv: (6222, 1504)
[SAVE] dfA_DKI_12_10.csv: (8174, 1136)
[SAVE] dfB_DKI_12_10.csv: (8174, 1136)
[SAVE] dfA_DKI_cog_12_10.csv: (7638, 1140)
[SAVE] dfB_DKI_cog_12_10.cs